# Multi-Object Detection & Persistent ID Tracking Pipeline

## Step-by-Step Walkthrough

This notebook demonstrates the complete pipeline:
1. **Video Loading** — Read and inspect the input video
2. **Detection** — Run YOLOv11 on individual frames
3. **Tracking** — Assign persistent IDs with BoT-SORT
4. **Annotation** — Draw bounding boxes, labels, trajectory trails
5. **Full Pipeline** — Process the entire video end-to-end
6. **Enhancements** — Generate heatmaps, object count charts

In [ ]:
import sys
sys.path.insert(0, "..")

import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image
import supervision as sv

from config.settings import PipelineConfig, DetectorConfig, TrackerConfig, AnnotatorConfig
from src.video_io import VideoReader, VideoWriter
from src.detector import Detector
from src.tracker import Tracker
from src.annotator import Annotator

%matplotlib inline
plt.rcParams["figure.figsize"] = (14, 8)

## 1. Load and Inspect Video

In [ ]:
VIDEO_PATH = "../data/input/cricket.mp4"  # Update this path as needed

reader = VideoReader(VIDEO_PATH)
print(f"Resolution: {reader.width}x{reader.height}")
print(f"FPS: {reader.fps}")
print(f"Total frames: {reader.total_frames}")
print(f"Duration: {reader.total_frames / reader.fps:.1f} seconds")

# Display a sample frame
sample_frame = reader.read_frame(0)
plt.imshow(cv2.cvtColor(sample_frame, cv2.COLOR_BGR2RGB))
plt.title("First Frame")
plt.axis("off")
plt.show()

## 2. Object Detection on a Single Frame

We use YOLOv11m pretrained on COCO, filtering for class 0 (person).

In [ ]:
# Initialize detector
det_config = DetectorConfig(model_name="yolo11m.pt", confidence=0.25, device="mps")
detector = Detector(det_config)
print(f"Detector loaded: {det_config.model_name} on {detector.device}")

# Run detection on frame 0
frame = reader.read_frame(0)
detections = detector.detect(frame)

print(f"\nDetected {len(detections)} persons")
print(f"Confidence scores: {detections.confidence}")

# Visualize detections
box_annotator = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_scale=0.5)

labels = [f"Person ({conf:.2f})" for conf in detections.confidence]
annotated = box_annotator.annotate(scene=frame.copy(), detections=detections)
annotated = label_annotator.annotate(scene=annotated, detections=detections, labels=labels)

plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.title(f"Detection Results: {len(detections)} persons found")
plt.axis("off")
plt.show()

## 3. Multi-Object Tracking on a Short Sequence

We process 30 consecutive frames to demonstrate tracking with persistent IDs.
BoT-SORT uses ReID embeddings + camera motion compensation to maintain stable IDs.

In [ ]:
# Initialize tracker
trk_config = TrackerConfig(tracker_type="botsort")
tracker = Tracker(trk_config)
print(f"Tracker: {trk_config.tracker_type}")
print(f"ReID model: {trk_config.reid_model}")
print(f"CMC method: {trk_config.cmc_method}")

# Process first 30 frames
ann_config = AnnotatorConfig(show_trajectory=True, trail_length=30)
annotator = Annotator(ann_config, fps=reader.fps)

tracked_frames = []
all_ids = set()

for i, frame in enumerate(reader):
    if i >= 30:
        break
    
    detections = detector.detect(frame)
    tracked = tracker.update(detections, frame)
    annotated = annotator.draw(frame, tracked, i)
    tracked_frames.append(annotated)
    
    if tracked.tracker_id is not None:
        for tid in tracked.tracker_id:
            all_ids.add(int(tid))

print(f"\nProcessed 30 frames")
print(f"Unique IDs assigned: {len(all_ids)}")
print(f"IDs: {sorted(all_ids)}")

# Display frames at intervals
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
display_frames = [0, 5, 10, 15, 20, 29]
for ax, idx in zip(axes.flat, display_frames):
    ax.imshow(cv2.cvtColor(tracked_frames[idx], cv2.COLOR_BGR2RGB))
    ax.set_title(f"Frame {idx}")
    ax.axis("off")
plt.suptitle("Tracking Results - First 30 Frames", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Full Pipeline Run

Now we run the complete pipeline on the entire video, producing an annotated output.

In [ ]:
from src.pipeline import TrackingPipeline

config = PipelineConfig(
    input_path=VIDEO_PATH,
    output_dir="../data/output",
    screenshot_dir="../data/screenshots",
    screenshot_interval=100,
)

pipeline = TrackingPipeline(config)
results = pipeline.run()

print(f"\nPipeline Results:")
print(f"  Total frames:       {results.total_frames}")
print(f"  Processing FPS:     {results.processing_fps:.1f}")
print(f"  Total unique IDs:   {results.total_unique_ids}")
print(f"  Avg tracked/frame:  {results.avg_detections_per_frame:.1f}")
print(f"  Output video:       {config.output_video_path}")

## 5. Enhancements

### 5a. Movement Heatmap

In [ ]:
from enhancements.heatmap import HeatmapGenerator

# Generate heatmap from pipeline results
heatmap_gen = HeatmapGenerator(
    resolution=(reader.width, reader.height),
    blur_sigma=30,
    alpha=0.6,
)
heatmap_gen.accumulate_from_results(results.track_positions)

# Use first frame as background
bg_frame = reader.read_frame(0)
heatmap_img = heatmap_gen.generate(bg_frame)

# Save and display
heatmap_path, plt_path = heatmap_gen.save(
    "../report/figures/movement_heatmap.png", bg_frame,
    title="Player Movement Heatmap"
)

plt.imshow(cv2.cvtColor(heatmap_img, cv2.COLOR_BGR2RGB))
plt.title("Movement Heatmap - Where Players Spend the Most Time")
plt.axis("off")
plt.show()
print(f"Saved to: {heatmap_path}")

### 5b. Object Count Over Time

In [ ]:
from enhancements.object_count import ObjectCountChart

count_chart = ObjectCountChart(fps=reader.fps)
chart_path = count_chart.generate(
    results.detections_per_frame,
    "../report/figures/object_count.png",
    title="Tracked Players Over Time"
)

stats = count_chart.generate_summary_stats(results.detections_per_frame)
print("Detection Statistics:")
for k, v in stats.items():
    print(f"  {k}: {v}")

# Display the chart (matplotlib already saved it)
from IPython.display import Image as IPImage
IPImage(filename=chart_path)

### 5c. Model Comparison

Compare YOLOv11 nano/medium/large with BoT-SORT and ByteTrack.
This runs on the first 300 frames for speed.

In [ ]:
from enhancements.model_comparison import ModelComparison

comparison = ModelComparison(config)
comp_results = comparison.run_comparison(
    models=["yolo11n.pt", "yolo11m.pt", "yolo11l.pt"],
    trackers=["botsort", "bytetrack"],
    max_frames=300,
)

# Generate comparison charts
charts = comparison.generate_charts("../report/figures")
json_path = comparison.save_results("../report/figures")

print(f"\nComparison Results:")
print(f"{'Model':<15} {'Tracker':<12} {'FPS':>8} {'Unique IDs':>12} {'Avg Det/Frame':>15}")
print("-" * 65)
for r in comp_results:
    print(f"{r.model_name:<15} {r.tracker_type:<12} {r.processing_fps:>8.1f} {r.total_unique_ids:>12} {r.avg_detections_per_frame:>15.1f}")

# Display chart
IPImage(filename=charts[0])